In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "raw"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print(train.shape, test.shape)
train.head()

In [ ]:
missing = pd.DataFrame({
    "train_missing": train.isna().sum(),
    "train_%": (train.isna().mean() * 100).round(1),
    "test_missing": test.isna().sum(),
})
missing[missing.sum(axis=1) > 0]

In [ ]:
train["Survived"].value_counts(normalize=True).round(3)

In [ ]:
for col in ["Sex", "Pclass", "Embarked"]:
    print(train.groupby(col)["Survived"].mean().round(3), "\n")

In [ ]:
pd.crosstab(train["Pclass"], train["Sex"], values=train["Survived"], aggfunc="mean").round(2)

# Titanic: exploratory data analysis

## Findings
1. **Missing data:** Age is missing for 19.9% of passengers, Cabin for 77.1% and Embarked for 2 rows. test.csv also has 1 missing Fare, so the pipeline must handle gaps that never appear in training.
2. **Sex:** women survived at 74.2% vs 18.9% for men. This is the strongest single signal.
3. **Class:** 1st class survived at 63.0% vs 24.2% in 3rd. Even women in 3rd class only survived at 50%, while nearly all women in 1st and 2nd class survived (97% and 92%).
4. **Titles:** "Master" (young boys) survived at 57%, far above "Mr" at 16%. Sex alone can't capture this, and titles also hint at age where Age is missing.
5. **Family size:** passengers travelling alone survived at 30%, small families (2–4 people) did best at 55–72%, and large families (5+) did worst at 0–33%.

## Baselines
- Everyone died: 0.616
- Women survive: 0.787 (the score to beat)

## Decisions for the pipeline
- Fill Age and Fare with the median, and Embarked with the most common port.
- Replace Cabin with a yes/no `HasCabin` feature, because 77% of it is missing.
- Keep Sex, Pclass and Embarked; one-hot encode the categories.
- Add `Title` (rare titles grouped as "Rare"), `FamilySize` and `IsAlone`.
- Drop Name, Ticket, Cabin and PassengerId as raw columns.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
train.loc[train["Survived"] == 0, "Age"].plot.hist(bins=30, alpha=0.6, label="Died", ax=ax)
train.loc[train["Survived"] == 1, "Age"].plot.hist(bins=30, alpha=0.6, label="Survived", ax=ax)
ax.set_xlabel("Age")
ax.legend()
plt.show()

In [ ]:
titles = train["Name"].str.extract(r",\s*([^.]+)\.")[0]
pd.DataFrame({"count": titles.value_counts(),
              "survival": train.groupby(titles)["Survived"].mean().round(2)})

In [ ]:
family_size = train["SibSp"] + train["Parch"] + 1
train.groupby(family_size)["Survived"].agg(["count", "mean"]).round(2)

In [ ]:
all_died = (train["Survived"] == 0).mean()
women_survive = ((train["Sex"] == "female").astype(int) == train["Survived"]).mean()
print(f"Baseline 'everyone died':  {all_died:.3f}")
print(f"Baseline 'women survive':  {women_survive:.3f}")